# Kata 15 — Auto-corrección Numérica

> Spec: [`specs/015-self-correction/spec.md`](../../specs/015-self-correction/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap

client, settings = bootstrap(model="claude-sonnet-4-6", budget_calls=6)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Cuando el modelo extrae números (totales, sumas), debe verificar lo declarado contra lo calculado. Si difieren más allá de un epsilon, marcar `mismatch=true` y enrutar a humano. Nunca elegir silenciosamente.

## 2. Por qué importa

Una factura puede tener un total declarado distinto al sumatorio de sus líneas — por OCR, por error humano o por fraude. Tomar uno de los dos sin avisar oculta el problema.

## 3. Ejemplo correcto — schema con cross-check tipado

In [ ]:
import json

EXTRACT_INVOICE = {
    "name": "extract_invoice",
    "description": "Extrae factura con verificación cruzada.",
    "input_schema": {
        "type": "object",
        "properties": {
            "invoice_id": {"type": "string"},
            "line_items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string"},
                        "amount": {"type": "number"},
                    },
                    "required": ["description","amount"],
                },
            },
            "stated_total": {"type": "number"},
            "computed_total": {"type": "number"},
            "mismatch": {"type": "boolean"},
            "delta": {"type": "number"},
            "needs_human_review": {"type": "boolean"},
        },
        "required": ["invoice_id","line_items","stated_total","computed_total","mismatch","delta","needs_human_review"],
    },
}

DOC_OK = '''Factura INV-100
- Servicio A: $100.00
- Servicio B: $250.00
- Servicio C: $50.00
TOTAL: $400.00'''

DOC_BAD = '''Factura INV-101
- Producto X: $1234.56
- Producto Y: $200.00
TOTAL: $1500.00'''   # debería ser $1434.56

def extract(client, doc: str):
    system = (
        "Extrae líneas y totales de la factura. Calcula `computed_total` sumando las líneas; "
        "si difiere de `stated_total` por más de 0.01, marca `mismatch=true` y `needs_human_review=true`. "
        "Nunca corrijas silenciosamente."
    )
    resp = client.messages.create(
        system=system,
        messages=[{"role":"user","content": doc}],
        tools=[EXTRACT_INVOICE],
        tool_choice={"type":"tool","name":"extract_invoice"},
    )
    return next(b.input for b in resp.content if b.type == "tool_use")

print("OK:", json.dumps(extract(client, DOC_OK), indent=2, ensure_ascii=False))
print("\nMISMATCH:", json.dumps(extract(client, DOC_BAD), indent=2, ensure_ascii=False))

## 4. Anti-patrón — confiar ciegamente en `stated_total`

In [ ]:
SIMPLE_TOOL = {
    "name": "extract_simple",
    "description": "Extrae factura.",
    "input_schema": {
        "type": "object",
        "properties": {
            "invoice_id": {"type": "string"},
            "total": {"type": "number"},
        },
        "required": ["invoice_id","total"],
    },
}

def extract_simple(client, doc):
    resp = client.messages.create(
        system="Extrae el id y el total de la factura.",
        messages=[{"role":"user","content": doc}],
        tools=[SIMPLE_TOOL],
        tool_choice={"type":"tool","name":"extract_simple"},
    )
    return next(b.input for b in resp.content if b.type == "tool_use")

print("anti-patrón en DOC_BAD:", extract_simple(client, DOC_BAD))

El extractor simple devuelve `total: 1500.00` sin saber que las líneas suman 1434.56. El consumidor downstream usa el total declarado, el delta de 65.44 se cuela invisible.

## 5. Argumento de certificación

- **Dos fuentes de verdad** (declarada vs computada). Deben cumplir un invariante (`|delta| < ε`).
- **`mismatch` + `needs_human_review`** son campos *required*: el modelo no puede omitirlos.
- **Nunca corregir silenciosamente.** Escalar o devolver con flag.
- **Conexión con otros katas**: el flag de `needs_human_review` es la entrada del Kata 16 (handoff); la cadena de extracciones encaja con Kata 12; las cifras conservan provenance del documento (Kata 20).

## 6. Auto-evaluación

**1. ¿Qué pasa si el documento no declara total (sólo línea por línea)?**

`stated_total = null` (campo nullable). `computed_total` se calcula. `mismatch = false` (no hay con qué comparar). `needs_human_review = false`. Estado limpio: el agente reporta sólo lo computado.

**2. ¿Cómo distingo "error de OCR" de "fraude" en el flag?**

A nivel del flag, no se distinguen — ambos disparan `needs_human_review`. La diferenciación es de proceso: la cola humana tiene heurísticas (delta pequeño y consistente con redondeos vs delta grande sin patrón).

**3. ¿Qué prueba reintroduce el anti-patrón (auto-corrección silenciosa) y qué assert falla?**

Modificar el system prompt a "si difieren, devuelve `stated_total` y no menciones la diferencia". Una aserción defensiva en el cliente: si `stated_total` y `computed_total` están ambos presentes y difieren, `mismatch` debe ser `true`. Si el modelo pone `false` con valores discrepantes, falla el assert.